# Risk-averse mutation analysis — choice-evolution scatter maps

This notebook produces focused scatter maps for selected risk settings and mutation families. It is intentionally separate from the aggregate notebook because the map layout is controlled by match-level plot variables and can become crowded when all risk settings are shown at once.

Recommended workflow:

1. Run `risk_averse_mutation_01_aggregate_analysis.ipynb` first.
2. Use this notebook for one mutation family and one risk mode or a small set of risk labels at a time.
3. Export compact match-level choice tables for manuscript diagnostics.

In [ ]:
# ============================================================
# SECTION 1. USER CONTROLS
# ============================================================
from pathlib import Path

# Path controls. Leave as None for the standard Code_Submission/simulation_mutation_averse layout.
code_root_override = None
risk_averse_mutation_dir_override = None
risk_output_base_dir_override = None
risk_averse_mutation_folder_name = "simulation_mutation_averse"
risk_output_base_folder_name = "Output files (Risk Averse, Mutation, Verified)"
analysis_output_folder_name = "Risk_Averse_Mutation_Analysis"

# Plot content controls.
# Use one family at a time for journal-readable panels.
mutation_family_to_plot = "cannibalization"   # "shape", "basis", "load_price", "cannibalization", or "all"
scatter_map_kind = "correlation"              # "correlation", "mean", or "median"
include_baseline_panel = True

# Risk filters. Leave as None to keep all; use one of these filters to avoid overcrowded figures.
filter_risk_labels = None                      # e.g. ["joint_low", "joint_medium"]
filter_risk_modes = None     # e.g. ["Seller risk-averse"], ["Buyer risk-averse"], None
filter_target_physical_shifts = None           # e.g. [-0.10, -0.30, -0.50]
filter_match_ids = None
filter_starting_baseline_profiles = None       # e.g. ["Fix"], ["AsC"], None
filter_current_profiles = None                 # e.g. ["No Contract"], None

# Axis data controls.
# "simulated" is usually the correct choice for mutated samples.
axis_source = "simulated"                      # "simulated", "ref", or "original"
common_axis_limits_across_panels = True
axis_padding_fraction = 0.08
manual_xlim = None                             # e.g. (-1, 1)
manual_ylim = None

# Output controls.
save_choice_table = True
make_switch_bar_figure = True

In [ ]:
# ============================================================
# SECTION 2. FIGURE CONTROLS
# ============================================================
make_figures = True
save_png = True
save_pdf = False
save_svg = False
display_figures_in_notebook = True
figure_dpi = 600

# Layout controls.
max_risk_rows_per_figure = 6
fig_width_per_panel = 4.8
fig_height_per_risk_row = 4.2
minimum_fig_width = 10.0
minimum_fig_height = 4.8
panel_title_fontsize = 12
axis_label_fontsize = 11
tick_label_fontsize = 10
legend_fontsize = 9

# Point controls.
marker_size = 42
baseline_marker_size = 26
point_alpha = 0.82
baseline_alpha = 0.30
show_match_id_labels = False
match_id_label_fontsize = 7
jitter_points = False
jitter_scale = 0.003

# Visual grammar. Marker encodes PPA type; color encodes selected profile.
profile_order = ["Fix", "AsC", "AsG", "No Contract", "Unknown"]
ppa_type_marker_map = {"Physical": "o", "Virtual": "^", "No Contract": "X", "Unknown": "s"}
profile_color_map = {
    "Fix": "C0",
    "AsC": "C1",
    "AsG": "C2",
    "No Contract": "C3",
    "Unknown": "C7",
}

family_display_labels = {
    "shape": "Profile-shape deterioration",
    "basis": "Basis deterioration",
    "load_price": "Buyer load-price intensification",
    "cannibalization": "Seller-side cannibalization",
    "baseline": "Baseline",
}
risk_mode_order = ["Seller risk-averse", "Buyer risk-averse", "Joint risk-averse", "Risk neutral", "Other"]

In [ ]:
# ============================================================
# SECTION 3. IMPORTS, PATHS, AND TABLE LOADING
# ============================================================
from __future__ import annotations

import math
import re
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except Exception:
    display = print

NOTEBOOK_CWD = Path.cwd().resolve()


def _as_path_or_none(value) -> Optional[Path]:
    if value is None:
        return None
    text = str(value).strip()
    if text == "" or text.lower() in {"none", "nan", "<na>"}:
        return None
    return Path(text).expanduser().resolve()


def _dedupe_paths(paths: Iterable[Path]) -> list[Path]:
    out, seen = [], set()
    for p in paths:
        try:
            rp = p.expanduser().resolve()
        except Exception:
            continue
        if str(rp) not in seen:
            out.append(rp)
            seen.add(str(rp))
    return out


def resolve_code_root() -> Path:
    override = _as_path_or_none(code_root_override)
    if override is not None:
        return override
    if NOTEBOOK_CWD.name == risk_averse_mutation_folder_name:
        return NOTEBOOK_CWD.parent
    if (NOTEBOOK_CWD / risk_averse_mutation_folder_name).exists():
        return NOTEBOOK_CWD
    if (NOTEBOOK_CWD.parent / risk_averse_mutation_folder_name).exists():
        return NOTEBOOK_CWD.parent
    return NOTEBOOK_CWD


def resolve_ra_dir(code_root: Path) -> Path:
    override = _as_path_or_none(risk_averse_mutation_dir_override)
    if override is not None:
        return override
    candidates = [NOTEBOOK_CWD, code_root / risk_averse_mutation_folder_name, NOTEBOOK_CWD / risk_averse_mutation_folder_name, NOTEBOOK_CWD.parent / risk_averse_mutation_folder_name]
    for c in _dedupe_paths(candidates):
        if (c / risk_output_base_folder_name).exists():
            return c
    return (code_root / risk_averse_mutation_folder_name).resolve()


CODE_ROOT = resolve_code_root()
RA_DIR = resolve_ra_dir(CODE_ROOT)
RISK_OUTPUT_BASE = _as_path_or_none(risk_output_base_dir_override) or (RA_DIR / risk_output_base_folder_name).resolve()
ANALYSIS_DIR = RISK_OUTPUT_BASE / analysis_output_folder_name
TABLE_DIR = ANALYSIS_DIR / "tables"
FIGURE_DIR = ANALYSIS_DIR / "choice_evolution_figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

paired_table_path = TABLE_DIR / "RA_Mutation_Paired_Baseline_Changes_Long.csv"
standardized_table_path = TABLE_DIR / "RA_Mutation_Standardized_All_Rows.csv"


def _read_csv_if_exists(path: Path, **kwargs) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()
    try:
        if not path.exists():
            return pd.DataFrame()
        if path.stat().st_size == 0:
            print(f"Empty CSV (size=0): {path}")
            return pd.DataFrame()
        return read_csv_optimized(path, **kwargs)
    except pd.errors.EmptyDataError:
        print(f"EmptyDataError reading {path}; returning empty DataFrame")
        return pd.DataFrame()
    except Exception:
        print(f"Error reading {path}; re-raising")
        raise

missing = [p for p in (paired_table_path, standardized_table_path) if not p.exists()]
if missing:
    alt_candidates = []
    for root in _dedupe_paths([RISK_OUTPUT_BASE, CODE_ROOT, NOTEBOOK_CWD]):
        if root.exists():
            for name in ("RA_Mutation_Paired_Baseline_Changes_Long.csv", "RA_Mutation_Standardized_All_Rows.csv"):
                alt_candidates.extend(sorted(root.rglob(name)))
    alt_text = ""
    if alt_candidates:
        alt_text = "\nCandidate files found elsewhere:\n" + "\n".join(str(p) for p in sorted(set(alt_candidates)))
    raise FileNotFoundError(
        "This notebook expects the aggregate notebook outputs. Run "
        "risk_averse_mutation_01_aggregate_analysis.ipynb first, or place the expected tables under: "
        f"{TABLE_DIR}\n\nMissing files:\n" + "\n".join(str(p.name) for p in missing) + alt_text
    )

paired_df = _read_csv_if_exists(paired_table_path)
all_rows_df = _read_csv_if_exists(standardized_table_path)

if paired_df.empty:
    print(f"Paired table is empty: {paired_table_path}")
if all_rows_df.empty:
    print(f"All-rows table is empty: {standardized_table_path}")

print("Loaded paired table:", paired_table_path)
print("Loaded all-row table:", standardized_table_path)
print("Paired mutation rows:", len(paired_df))
print("All standardized rows:", len(all_rows_df))

In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


In [ ]:
# ============================================================
# SECTION 4. FILTERING AND AXIS FIELD HELPERS
# ============================================================

def _num(s):
    return pd.to_numeric(s, errors="coerce")


def normalize_profile_value(value) -> str:
    text = str(value).strip()
    low = text.lower().replace("_", " ").replace("-", " ")
    if low in {"fix", "fixed"}:
        return "Fix"
    if low in {"asc", "as contracted"}:
        return "AsC"
    if low in {"asg", "as generated"}:
        return "AsG"
    if low in {"no contract", "none", "", "nan", "na", "n/a"}:
        return "No Contract"
    return text


def normalize_family_value(value) -> str:
    text = str(value).strip()
    low = text.lower().replace("-", "_").replace(" ", "_")
    if low in family_display_labels:
        return family_display_labels[low]
    if "shape" in low:
        return family_display_labels["shape"]
    if "basis" in low:
        return family_display_labels["basis"]
    if "load" in low or "buyer" in low:
        return family_display_labels["load_price"]
    if "cannibal" in low or "seller" in low:
        return family_display_labels["cannibalization"]
    if "baseline" in low:
        return "Baseline"
    return text


def target_shift_label(value) -> str:
    if pd.isna(value):
        return "NA"
    v = float(value)
    if abs(v) < 1e-12:
        return "0"
    return f"{v:+.2f}"


def _numeric_filter(series, allowed, atol=1e-9):
    if allowed is None:
        return pd.Series(True, index=series.index)
    vals = [float(x) for x in (allowed if isinstance(allowed, (list, tuple, set)) else [allowed])]
    s = _num(series)
    keep = pd.Series(False, index=series.index)
    for v in vals:
        keep = keep | np.isclose(s, v, atol=atol)
    return keep


def axis_columns(kind: str, source: str) -> tuple[str, str, str, str]:
    source = source.lower()
    suffix_order = {
        "simulated": ["simulated", "ref", "original"],
        "ref": ["ref", "simulated", "original"],
        "original": ["original", "ref", "simulated"],
    }.get(source, ["simulated", "ref", "original"])

    candidates = {
        "correlation": [
            (f"shape_corr_{s}", f"basis_corr_{s}", "Profile-shape correlation", "Basis / nodal-price correlation") for s in suffix_order
        ],
        "mean": [
            (f"volume_mismatch_mean_{s}", f"price_spread_mean_{s}", "Mean volume-mismatch index", "Mean price-spread index") for s in suffix_order
        ],
        "median": [
            (f"volume_mismatch_median_{s}", f"price_spread_median_{s}", "Median volume-mismatch index", "Median price-spread index") for s in suffix_order
        ],
    }
    for xcol, ycol, xlabel, ylabel in candidates[kind]:
        if xcol in all_rows_df.columns and ycol in all_rows_df.columns:
            return xcol, ycol, xlabel, ylabel
    raise KeyError(f"No axis columns found for kind={kind!r}, source={source!r}.")


def prepare_choice_table() -> pd.DataFrame:
    xcol, ycol, xlabel, ylabel = axis_columns(scatter_map_kind, axis_source)
    cols = [
        "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity", "risk_setting_order",
        "match_id", "scenario_name", "scenario_type", "mutation_family", "mutation_family_normalized",
        "target_shift_signed", "target_shift_label", "ppa_type_normalized", "selected_profile_for_switch",
        xcol, ycol,
    ]
    cols = [c for c in cols if c in all_rows_df.columns]
    out = all_rows_df[cols].copy()
    out["_x"] = _num(out[xcol])
    out["_y"] = _num(out[ycol])
    out["_axis_x_label"] = xlabel
    out["_axis_y_label"] = ylabel
    out["target_shift_label"] = out["target_shift_signed"].map(target_shift_label)
    return out


def apply_choice_filters(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if filter_risk_labels is not None:
        out = out.loc[out["risk_label"].astype(str).isin({str(x) for x in filter_risk_labels})].copy()
    if filter_risk_modes is not None:
        out = out.loc[out["risk_mode"].astype(str).isin({str(x) for x in filter_risk_modes})].copy()
    if filter_match_ids is not None:
        out = out.loc[out["match_id"].astype(int).isin({int(x) for x in filter_match_ids})].copy()
    if mutation_family_to_plot != "all":
        fam = normalize_family_value(mutation_family_to_plot)
        is_baseline = out["scenario_type"].astype(str).eq("baseline")
        out = out.loc[is_baseline | out["mutation_family_normalized"].eq(fam)].copy()
    if filter_target_physical_shifts is not None:
        is_baseline = out["scenario_type"].astype(str).eq("baseline")
        out = out.loc[is_baseline | _numeric_filter(out["target_shift_signed"], filter_target_physical_shifts)].copy()
    if not include_baseline_panel:
        out = out.loc[~out["scenario_type"].astype(str).eq("baseline")].copy()

    if filter_starting_baseline_profiles is not None:
        keep_profiles = {normalize_profile_value(x) for x in filter_starting_baseline_profiles}
        base = paired_df[["risk_setting_key", "match_id", "baseline_selected_profile_for_switch"]].drop_duplicates() if "risk_setting_key" in paired_df.columns else pd.DataFrame()
        if not base.empty and "risk_setting_key" in out.columns:
            out = out.merge(base, on=["risk_setting_key", "match_id"], how="left")
            out = out.loc[out["baseline_selected_profile_for_switch"].map(normalize_profile_value).isin(keep_profiles)].copy()
    if filter_current_profiles is not None:
        keep_profiles = {normalize_profile_value(x) for x in filter_current_profiles}
        out = out.loc[out["selected_profile_for_switch"].map(normalize_profile_value).isin(keep_profiles)].copy()
    return out.reset_index(drop=True)


choice_df = apply_choice_filters(prepare_choice_table())
if choice_df.empty:
    raise ValueError("No rows remain after choice-map filters.")

if save_choice_table:
    out_path = TABLE_DIR / f"RA_Mutation_Choice_Map_Table__{mutation_family_to_plot}__{scatter_map_kind}.csv"
    choice_df.to_csv(out_path, index=False)
    print("Saved compact choice-map table:", out_path)

print("Rows for choice maps:", len(choice_df))
print("Risk labels:", choice_df["risk_label"].nunique())
print("Scenarios:", choice_df["scenario_name"].nunique())

In [ ]:
# ============================================================
# SECTION 5. PLOTTING HELPERS
# ============================================================

def _safe_stem(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(text)).strip("_")


def _save_figure(fig, stem: str):
    saved = []
    if save_png:
        path = FIGURE_DIR / f"{stem}.png"
        fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
        saved.append(path)
    if save_pdf:
        path = FIGURE_DIR / f"{stem}.pdf"
        fig.savefig(path, bbox_inches="tight")
        saved.append(path)
    if save_svg:
        path = FIGURE_DIR / f"{stem}.svg"
        fig.savefig(path, bbox_inches="tight")
        saved.append(path)
    if saved:
        print("Saved", stem, "->", ", ".join(p.name for p in saved))


def _close_or_show(fig):
    if display_figures_in_notebook:
        display(fig)
    plt.close(fig)


def ordered_risks(df: pd.DataFrame) -> list[str]:
    cols = ["risk_label_display", "risk_mode", "risk_intensity", "risk_setting_order"]
    existing = [c for c in cols if c in df.columns]
    rows = df[existing].drop_duplicates().copy()
    if "risk_mode" not in rows.columns:
        rows["risk_mode"] = "Other"
    if "risk_intensity" not in rows.columns:
        rows["risk_intensity"] = 0
    if "risk_setting_order" not in rows.columns:
        rows["risk_setting_order"] = 0
    mode_rank = {m: i for i, m in enumerate(risk_mode_order)}
    rows["_mode_rank"] = rows["risk_mode"].map(lambda x: mode_rank.get(x, 999))
    rows = rows.sort_values(["_mode_rank", "risk_intensity", "risk_setting_order", "risk_label_display"])
    return rows["risk_label_display"].tolist()


def ordered_scenarios(df: pd.DataFrame) -> list[str]:
    meta = df[["scenario_name", "scenario_type", "mutation_family_normalized", "target_shift_signed", "target_shift_label"]].drop_duplicates().copy()
    meta["_is_baseline"] = meta["scenario_type"].astype(str).eq("baseline").astype(int)
    meta["_shift"] = _num(meta["target_shift_signed"]).fillna(0)
    meta = meta.sort_values(["_is_baseline", "mutation_family_normalized", "_shift", "scenario_name"])
    # Baseline first if included.
    baseline = meta.loc[meta["scenario_type"].astype(str).eq("baseline"), "scenario_name"].tolist()
    mutation = meta.loc[~meta["scenario_type"].astype(str).eq("baseline"), "scenario_name"].tolist()
    return baseline + mutation


def scenario_title(sub: pd.DataFrame) -> str:
    if sub.empty:
        return ""
    row = sub.iloc[0]
    if str(row.get("scenario_type", "")) == "baseline":
        return "Baseline"
    fam = str(row.get("mutation_family_normalized", ""))
    shift = str(row.get("target_shift_label", ""))
    return f"{fam}\nshift {shift}"


def common_limits(df: pd.DataFrame):
    if manual_xlim is not None and manual_ylim is not None:
        return manual_xlim, manual_ylim
    x = _num(df["_x"]).replace([np.inf, -np.inf], np.nan).dropna()
    y = _num(df["_y"]).replace([np.inf, -np.inf], np.nan).dropna()
    def lim(s):
        if s.empty:
            return None
        lo, hi = float(s.min()), float(s.max())
        if math.isclose(lo, hi):
            pad = 0.05 if lo == 0 else abs(lo) * 0.05
        else:
            pad = (hi - lo) * axis_padding_fraction
        return (lo - pad, hi + pad)
    return manual_xlim or lim(x), manual_ylim or lim(y)


def _maybe_jitter(values, scale):
    arr = np.asarray(values, dtype=float)
    if not jitter_points:
        return arr
    rng = np.random.default_rng(12345)
    return arr + rng.normal(0.0, scale, size=len(arr))


def legend_handles():
    profile_handles = [Line2D([0], [0], marker="o", linestyle="", label=p, color=profile_color_map.get(p, "C7"), markersize=7) for p in profile_order]
    ppa_handles = [Line2D([0], [0], marker=m, linestyle="", label=p, color="black", markersize=7) for p, m in ppa_type_marker_map.items()]
    return profile_handles, ppa_handles

In [ ]:
# ============================================================
# SECTION 6. CHOICE-EVOLUTION SCATTER MAPS
# ============================================================

def plot_choice_evolution_maps(df: pd.DataFrame):
    risks = ordered_risks(df)
    scenarios = ordered_scenarios(df)
    if not risks or not scenarios:
        print("Skipped choice maps: no risks or scenarios.")
        return

    xlim, ylim = common_limits(df) if common_axis_limits_across_panels else (None, None)
    xlabel = str(df["_axis_x_label"].iloc[0])
    ylabel = str(df["_axis_y_label"].iloc[0])

    # Split many risk rows into multiple figures.
    for start in range(0, len(risks), max_risk_rows_per_figure):
        risk_batch = risks[start:start + max_risk_rows_per_figure]
        nrows = len(risk_batch)
        ncols = len(scenarios)
        fig_w = max(minimum_fig_width, fig_width_per_panel * ncols)
        fig_h = max(minimum_fig_height, fig_height_per_risk_row * nrows)
        fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), dpi=figure_dpi, squeeze=False)

        for r, risk_label in enumerate(risk_batch):
            for c, scenario in enumerate(scenarios):
                ax = axes[r, c]
                sub = df.loc[(df["risk_label_display"].astype(str).eq(str(risk_label))) & (df["scenario_name"].astype(str).eq(str(scenario)))].copy()
                if sub.empty:
                    ax.axis("off")
                    continue
                is_baseline = sub["scenario_type"].astype(str).eq("baseline").all()
                size = baseline_marker_size if is_baseline else marker_size
                alpha = baseline_alpha if is_baseline else point_alpha
                for ppa_type, ps in sub.groupby("ppa_type_normalized", dropna=False):
                    for profile, gs in ps.groupby("selected_profile_for_switch", dropna=False):
                        ax.scatter(
                            _maybe_jitter(gs["_x"], jitter_scale),
                            _maybe_jitter(gs["_y"], jitter_scale),
                            s=size,
                            marker=ppa_type_marker_map.get(str(ppa_type), ppa_type_marker_map["Unknown"]),
                            color=profile_color_map.get(str(profile), profile_color_map["Unknown"]),
                            alpha=alpha,
                            edgecolors="black",
                            linewidths=0.3,
                        )
                if show_match_id_labels:
                    for _, row in sub.iterrows():
                        ax.text(row["_x"], row["_y"], str(int(row["match_id"])), fontsize=match_id_label_fontsize, alpha=0.75)
                if common_axis_limits_across_panels:
                    if xlim is not None:
                        ax.set_xlim(xlim)
                    if ylim is not None:
                        ax.set_ylim(ylim)
                ax.axhline(0, linewidth=0.7, alpha=0.5)
                ax.axvline(0, linewidth=0.7, alpha=0.5)
                ax.grid(True, alpha=0.2)
                if r == 0:
                    ax.set_title(scenario_title(sub), fontsize=panel_title_fontsize)
                if c == 0:
                    ax.set_ylabel(f"{risk_label}\n{ylabel}", fontsize=axis_label_fontsize)
                else:
                    ax.set_ylabel("")
                if r == nrows - 1:
                    ax.set_xlabel(xlabel, fontsize=axis_label_fontsize)
                ax.tick_params(labelsize=tick_label_fontsize)

        profile_handles, ppa_handles = legend_handles()
        fig.legend(handles=profile_handles, loc="upper center", bbox_to_anchor=(0.5, 1.015), ncol=len(profile_handles), fontsize=legend_fontsize, frameon=True, title="Selected profile")
        fig.legend(handles=ppa_handles, loc="lower center", bbox_to_anchor=(0.5, -0.01), ncol=len(ppa_handles), fontsize=legend_fontsize, frameon=True, title="PPA type")
        fig.tight_layout(rect=[0, 0.03, 1, 0.97])
        stem = f"fig_choice_evolution_{_safe_stem(mutation_family_to_plot)}_{scatter_map_kind}_risks_{start + 1}_to_{start + len(risk_batch)}"
        _save_figure(fig, stem)
        _close_or_show(fig)


if make_figures:
    plot_choice_evolution_maps(choice_df)

In [ ]:
# ============================================================
# SECTION 7. OPTIONAL SWITCH-BAR FIGURE
# ============================================================

def plot_switch_bars(paired: pd.DataFrame):
    if not make_switch_bar_figure:
        return
    sub = paired.copy()
    if filter_risk_labels is not None:
        sub = sub.loc[sub["risk_label"].astype(str).isin({str(x) for x in filter_risk_labels})].copy()
    if filter_risk_modes is not None:
        sub = sub.loc[sub["risk_mode"].astype(str).isin({str(x) for x in filter_risk_modes})].copy()
    if mutation_family_to_plot != "all":
        sub = sub.loc[sub["mutation_family_normalized"].eq(normalize_family_value(mutation_family_to_plot))].copy()
    if filter_target_physical_shifts is not None:
        sub = sub.loc[_numeric_filter(sub["target_shift_signed"], filter_target_physical_shifts)].copy()
    if sub.empty:
        print("Skipped switch-bar figure: no paired rows after filters.")
        return
    grp = sub.groupby(["risk_label_display", "target_shift_signed", "target_shift_label"], dropna=False).agg(
        n_matches=("match_id", "count"),
        profile_switch_rate=("profile_changed", "mean"),
        contract_switch_rate=("contract_type_changed", "mean"),
        full_decision_switch_rate=("full_decision_changed", "mean"),
    ).reset_index()
    risks = ordered_risks(grp)
    nrows = len(risks)
    fig, axes = plt.subplots(nrows, 1, figsize=(default_fig_width if 'default_fig_width' in globals() else 12, max(4.5, 3.1 * nrows)), dpi=figure_dpi, squeeze=False)
    width = 0.25
    for r, risk in enumerate(risks):
        ax = axes[r, 0]
        rs = grp.loc[grp["risk_label_display"].eq(risk)].sort_values("target_shift_signed")
        x = np.arange(len(rs))
        ax.bar(x - width, rs["profile_switch_rate"], width=width, label="profile")
        ax.bar(x, rs["contract_switch_rate"], width=width, label="PPA type")
        ax.bar(x + width, rs["full_decision_switch_rate"], width=width, label="full decision")
        ax.set_xticks(x)
        ax.set_xticklabels(rs["target_shift_label"], fontsize=tick_label_fontsize)
        ax.set_ylabel("switch rate", fontsize=axis_label_fontsize)
        ax.set_title(risk, fontsize=panel_title_fontsize)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(True, axis="y", alpha=0.25)
        if r == 0:
            ax.legend(fontsize=legend_fontsize, frameon=True)
    axes[-1, 0].set_xlabel("Target physical shift", fontsize=axis_label_fontsize)
    fig.tight_layout()
    stem = f"fig_choice_switch_bars_{_safe_stem(mutation_family_to_plot)}_{scatter_map_kind}"
    _save_figure(fig, stem)
    _close_or_show(fig)


if make_figures:
    plot_switch_bars(paired_df)

print("Choice-evolution analysis complete.")
print(f"Figures: {FIGURE_DIR}")